In [1]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

model_path = "../models/biobert-ner-final"
label_names = ["O", "B-Disease", "I-Disease"]

model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded!")
print("Device:", device)

/Users/angelinagupta/biomedical-nlp-project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Model loaded!
Device: cpu


In [6]:
def predict_entities(text, model, tokenizer, label_names, device):
    # Tokenize input
    import re
    words = re.findall(r"\b\w+(?:'\w+)?\b", text)
    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)

    # Get predictions
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids = tokenized.word_ids()

    # Map predictions back to words
    word_predictions = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_predictions:
            word_predictions[word_id] = label_names[predictions[idx]]

    # Extract entity spans
    entities = []
    current_entity = []
    current_label = None

    for word_id, label in sorted(word_predictions.items()):
        word = words[word_id]
        if label == "B-Disease":
            if current_entity:
                entities.append(" ".join(current_entity))
            current_entity = [word]
            current_label = "Disease"
        elif label == "I-Disease" and current_entity:
            current_entity.append(word)
        else:
            if current_entity:
                entities.append(" ".join(current_entity))
                current_entity = []
                current_label = None

    if current_entity:
        entities.append(" ".join(current_entity))

    return {
        "text": text,
        "entities": entities,
        "word_labels": {words[k]: v for k, v in word_predictions.items()}
    }

print("Inference function ready!")

Inference function ready!


In [7]:
test_sentences = [
    "The patient was diagnosed with Alzheimer's disease and diabetes.",
    "Metformin is commonly used to treat type 2 diabetes mellitus.",
    "She suffered from acute myocardial infarction and hypertension.",
    "The child showed symptoms of autism spectrum disorder.",
    "BRCA1 mutations are associated with hereditary breast cancer."
]

for sentence in test_sentences:
    result = predict_entities(sentence, model, tokenizer, label_names, device)
    print(f"Text:     {result['text']}")
    print(f"Diseases: {result['entities']}")
    print()

Text:     The patient was diagnosed with Alzheimer's disease and diabetes.
Diseases: ["Alzheimer's disease", 'diabetes']

Text:     Metformin is commonly used to treat type 2 diabetes mellitus.
Diseases: ['type 2 diabetes mellitus']

Text:     She suffered from acute myocardial infarction and hypertension.
Diseases: ['acute myocardial infarction', 'hypertension']

Text:     The child showed symptoms of autism spectrum disorder.
Diseases: ['autism spectrum disorder']

Text:     BRCA1 mutations are associated with hereditary breast cancer.
Diseases: ['hereditary breast cancer']



In [8]:
def highlight_entities(text, model, tokenizer, label_names, device):
    result = predict_entities(text, model, tokenizer, label_names, device)
    words = text.split()
    highlighted = []
    
    for word in words:
        label = result["word_labels"].get(words.index(word), "O")
        if label in ["B-Disease", "I-Disease"]:
            highlighted.append(f"[{word}]")
        else:
            highlighted.append(word)
    
    print("Original:    ", text)
    print("Highlighted: ", " ".join(highlighted))
    print("Entities:    ", result["entities"])
    print()

# Test it
highlight_entities(
    "Mutations in PTEN cause Cowden syndrome and breast cancer.",
    model, tokenizer, label_names, device
)

highlight_entities(
    "The patient had both Parkinson disease and severe depression.",
    model, tokenizer, label_names, device
)

Original:     Mutations in PTEN cause Cowden syndrome and breast cancer.
Highlighted:  Mutations in PTEN cause Cowden syndrome and breast cancer.
Entities:     ['Cowden syndrome', 'breast cancer']

Original:     The patient had both Parkinson disease and severe depression.
Highlighted:  The patient had both Parkinson disease and severe depression.
Entities:     ['Parkinson disease', 'depression']



In [5]:
inference_code = '''
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

def load_ner_model(model_path):
    label_names = ["O", "B-Disease", "I-Disease"]
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    return model, tokenizer, label_names, device

def predict_entities(text, model, tokenizer, label_names, device):
    words = text.split()
    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids = tokenized.word_ids()

    word_predictions = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_predictions:
            word_predictions[word_id] = label_names[predictions[idx]]

    entities = []
    current_entity = []

    for word_id, label in sorted(word_predictions.items()):
        word = words[word_id]
        if label == "B-Disease":
            if current_entity:
                entities.append(" ".join(current_entity))
            current_entity = [word]
        elif label == "I-Disease" and current_entity:
            current_entity.append(word)
        else:
            if current_entity:
                entities.append(" ".join(current_entity))
                current_entity = []

    if current_entity:
        entities.append(" ".join(current_entity))

    return {"text": text, "entities": entities}
'''

with open("../src/ner_inference.py", "w") as f:
    f.write(inference_code)

print("Inference script saved to src/ner_inference.py")

Inference script saved to src/ner_inference.py
